# Step 8 checks: do the harmonised values agree across sites?

Reads the files written by `harmonise.ipynb` (`derived/harmonised/`) and compares sites. Nothing here changes the data. The checks look for units, definitions and placeholders that differ between sites, concepts that are duplicates of others, and the effect of the GOSH Epic switch. Counts, quantiles and names only; no minimum or maximum values. Flags are places to look, not errors.

Sections: 1 distributions across sites; 2 follow-up on flagged concepts; 3 one-site concepts against common ones; 4 fluid definitions; 5 GOSH before and after the Epic switch; 6 units and scale (age-adjusted); 7 RBH rows after the recorded end; 8 drug units; 9 save the report.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from datetime import date
from pathlib import Path

import pandas as pd

open_picu_dir = Path("/home/jormond/Documents/DS217/users/jormond/open_picu")
sys.path.insert(0, str(open_picu_dir / "code"))
import concepts as C
import diagnostics as D
import harmonise as H

In [ ]:
assembled_dir = open_picu_dir / "derived" / "assembled"
reports_dir = open_picu_dir / "reports"
harmonised_dir = open_picu_dir / "derived" / "harmonised"
static = pd.read_parquet(open_picu_dir / "derived" / "static" / "pooled_static.parquet")
today = date.today().isoformat()
pd.set_option("display.width", 200)

## 1. Distributions across sites
Per site and concept: quantiles of the harmonised values, the share of zero, negative and flagged rows. Then a side-by-side comparison with flags where the sites' medians, spreads, tails or zero shares differ. The first table lists the flagged concepts.

In [ ]:
dist = H.distribution_check(harmonised_dir)
cmp_ = H.compare_sites(dist)
print(len(cmp_), "concepts;", int((~cmp_["looks_ok"]).sum()), "flagged")
cmp_[~cmp_["looks_ok"]]

## 2. Follow-up on flagged concepts
Looks inside the concepts flagged above: the commonest high values (placeholders), the split by source table, the raw SMH I:E strings, and what the integers 11 to 19 at SMH mean (the ratio implied by the set rate and inspiratory time at the same chart time: a recorded "14" with an implied median near 1.4 is 1.4; near 4 it is 1:4). The list of checks is the one that followed from the first run; change it to follow the flags in section 1.

In [ ]:
print(H.source_check(harmonised_dir, "GOSH", "bg_venous_sat").to_string())
for site, concept, limit in [("GOSH", "insp_time", 5), ("GOSH", "albumin", 100), ("GOSH", "cvp", 40), ("GOSH", "potassium", 10)]:
    print(H.values_above(harmonised_dir, site, concept, limit).to_string(), "\n")

In [ ]:
ie = H.ie_check(harmonised_dir)
ie

## 3. Are the one-site concepts duplicates of common ones?
Each pair is matched by stay and hour. Near-identical values (correlation close to 1, differences close to 0) mean one of the pair is redundant and can be left out of the pooled release. This is the evidence for the `ONE_SITE_DROP` list in `anonymise.py`.

In [ ]:
pairs = pd.concat([
    H.pair_check(harmonised_dir, "GOSH", "hr", "pulse_rate", tol=3),
    H.pair_check(harmonised_dir, "GOSH", "resp_rate_total", "resp_rate_monitor", tol=2),
    H.pair_check(harmonised_dir, "SMH", "peep", "ps_peep", tol=1),
    H.pair_check(harmonised_dir, "SMH", "pip", "ps_peep", tol=2),
])
pairs

## 4. Fluid definitions: do the sites mean the same thing?
`output_total`, `intake_total` and `net_balance` differ across sites by large factors that no unit conversion explains. Some sources are hourly amounts, some are 24 h totals and some are running totals. The profile gives `rows_per_hour` (about 1 for an hourly record, well below 1 for a total recorded a few times a day), `median_day_per_kg` (for intake in children a few tens to about 150 ml/kg; a source about 24 times higher is a 24 h total repeated hourly) and `pct_nondecreasing` (about 50% for hourly amounts, near 100% for a running total). The second table checks, hour by hour, whether the concepts agree: net = intake - output, IV <= total intake, urine <= total output.

In [ ]:
profile, identities = H.fluid_definition_check(harmonised_dir, static)
display(profile.sort_values(["concept", "site", "source"]))
identities

## 5. GOSH before and after the Epic switch
Compares GOSH stays whose t0 is before 2019-04-20 with those on or after it, for every concept in the harmonised GOSH files. For each concept: the share of each era's stays that have it, the change in percentage points, the median per-stay mean value in each era, a standardised shift (post minus pre in units of the pre-era spread) and a KS distance. `flag` marks `absent after` (post share under a tenth of the pre share), `thin after` (under half), `new after`, and `values differ` (shift above 1 or KS above 0.3). Only concepts with at least 20% of stays in one era are flagged on coverage.

In [ ]:
era = H.gosh_era_comparison(harmonised_dir, static)
print(era[["n_pre", "n_post"]].iloc[0].to_dict())
era[era["flag"] != ""]

## 6. Units and scale across sites, age-adjusted
One row per numeric concept shared by at least two sites: median per site, crude and age-adjusted ratio of largest to smallest site median, and a unit-factor hint. `flag` = adjusted ratio above 1.3. This replaces the 3x screen in section 1, which would miss a 1.5 to 2x assay or unit difference. Creatinine was the one concept where the value gave the site away, so it is shown by age band: a unit or assay difference gives a constant ratio in every band; case mix shrinks once age is held fixed.

In [ ]:
uc = H.unit_consistency(harmonised_dir, static)
uc.to_csv(reports_dir / "unit_consistency.csv", index=False)
uc[["concept", "sites", "med_GOSH", "med_RBH", "med_SMH", "crude_ratio", "adjusted_ratio", "unit_hint", "flag"]]

In [ ]:
creat = H.concept_by_age(harmonised_dir, static, "creatinine")
creat.to_csv(reports_dir / "creatinine_by_age.csv", index=False)
creat

## 7. RBH: why do the vitals run past the recorded end?
Uses the ward times in the RBH vitals table: ward in and out relative to t0, whether a stay key has several ward episodes or admission keys, and how the last heart rate compares with the static end and with the ward out. The second table gives, for each clinical unit in the RBH vitals, the rows, the stays, and the percent of heart-rate rows more than 24 h before t0, from 24 h before to t0, inside the stay and after the static end.

In [ ]:
rbh_end = H.rbh_end_check(assembled_dir, static)
rbh_end

In [ ]:
rbh_units = H.rbh_unit_check(assembled_dir, static)
rbh_units

## 8. Drug units
Every unit string recorded for drugs at each site (only those seen in at least 10 stays), with what kind of value it is and, where the string parses, a factor that converts the recorded value to mg (or units) per hour, or per kg per hour. Nothing in the harmonised data is converted. Strings that do not parse have an empty factor; check those first.

In [ ]:
lookup = H.drug_unit_lookup(harmonised_dir)
lookup[lookup["factor"].isna()][["site", "unit", "rows", "stays", "kind"]]

## 9. Save the report
The tables above, counts and summaries only, in one file for review.

In [ ]:
D.write_report({
    "Concepts flagged in the cross-site comparison": cmp_[~cmp_["looks_ok"]],
    "I:E ratio at SMH, integers 11 to 19": ie,
    "One-site concepts against common ones": pairs,
    "Fluid definitions, profile by site and source": profile.sort_values(["concept", "site", "source"]),
    "Fluid definitions, identities": identities,
    "GOSH before and after the Epic switch (flagged concepts)": era[era["flag"] != ""],
    "Units and scale across sites": uc[["concept", "sites", "med_GOSH", "med_RBH", "med_SMH", "crude_ratio", "adjusted_ratio", "unit_hint", "flag"]],
    "Creatinine by age band": creat,
    "RBH vitals past the recorded end": rbh_end,
    "RBH vitals by clinical unit": rbh_units,
    "Drug unit strings that do not parse": lookup[lookup["factor"].isna()][["site", "unit", "rows", "stays", "kind"]],
}, reports_dir / f"Harmonise_checks_{today}.md", title="Harmonise checks")

In [ ]:
be = H.pair_check(harmonised_dir, "GOSH", "bg_base_excess", "bg_base_excess_recorded")
be